# Diagnose, Fix, Brief · The Meridian Pilot

A client's AI support agent is failing and they think it's the model. Your job is to find what's actually wrong, fix it, and prove the fix with numbers the client can take to their leadership. You can change anything around the model. You can't change the model itself.

## Before you start

You'll work this the way you'd work a stalled client system in real life: poke it, watch what it does, form a theory, change one thing, and see if the number moves.

About that number. Every time you run the scoreboard it doesn't run the system once. It runs the same support ticket five times and reports how often the agent actually resolved it. One clean run proves nothing. An agent's choices wobble from run to run, so you read the rate, not any single attempt. That rate, and the cost beside it, is your evaluation: the feedback you use to decide what to change and whether the change worked.

Here's what one run looks like, and what to watch:

```
==================================================================
  MERIDIAN SCOREBOARD  ·  claude-sonnet-5  ·  5 trials/ticket
==================================================================

  Ticket T-4471   routing: account (x4); account, billing (x1)
    sso_addressed          5/5  #####
    billing_resolved       1/5  #....
    no_false_resolution    1/5  #....
    no_overclaim           3/5  ###..
    --> RESOLVED in 1/5 trials

------------------------------------------------------------------
  RESOLVED  1/5 trials
  COST      $0.55 total  ·  $0.11/trial
------------------------------------------------------------------
```

A run only counts as resolved when every problem the customer raised actually got handled. That's a high bar, and it's deliberate. The four checks underneath are the breakdown: they point you at which part is holding the score down. And `routing` shows the call the coordinator made each time. When you see the same ticket routed different ways across five runs, that wobble is the whole reason one run can't tell you anything. Read the rate.

**Heads up on timing:** each scoreboard run takes a minute or two, since it's running the ticket five times in a row. Let it finish before you read the result.

The loop: run it, read the rate, go figure out why, change one thing, run it again. Watch the rate.

## Setup — connect to Claude

Run the next cell first. The setup cell creates a **`.env` file** the first time you run it (gitignored — your key is never committed). Open it, paste your key after `ANTHROPIC_API_KEY=`, save, and re-run — it survives kernel restarts, so you paste once. *(No `.env` yet? A hidden input box appears as a fallback.)* You're locked in when you see the green **"✓ API key verified"** banner. Red banner? Do what it says and run the cell again.

These cells run `{sys.executable}` (the notebook kernel's own Python) instead of plain `python`, so the script always uses the same environment the kernel installed `anthropic` into. Run this Setup cell first, it loads `sys` for the cells below.

In [ ]:
# Install dependencies into THIS kernel — safe to re-run; survives locked-down (PEP 668) Pythons.
import importlib.util, subprocess, sys

def _ensure_packages(requirements):
    """requirements: list of (import_name, pip_spec). Install only what is missing,
    into the running interpreter. Tries a normal install, then user-space, then a
    PEP 668 override (user-space first, system-wide only as a last resort). Every
    attempt is silent — pip's output is captured, not streamed — so a locked-down
    Python (Homebrew or Debian, PEP 668) no longer dumps a scary
    'externally-managed-environment' wall of text when a fallback is what actually
    succeeds. Only if every strategy fails does it surface the reason, with the
    venv fix instead of a raw traceback."""
    missing = [pip for mod, pip in requirements if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + " — first run only, please wait…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    last = None
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        last = subprocess.run(base + extra + missing, capture_output=True, text=True)
        if last.returncode == 0:
            return
    pip_said = (last.stderr or last.stdout or "").strip().splitlines() if last else []
    tail = "\n      ".join(pip_said[-3:]) if pip_said else "(no output from pip)"
    raise SystemExit(
        "\n  Couldn't install: " + ", ".join(missing) + "\n"
        "  This Python is locked down (PEP 668) or offline. Quickest fix is a venv:\n"
        f"      {sys.executable} -m venv .venv\n"
        "      source .venv/bin/activate          # Windows: see SETUP.md\n"
        f"      pip install {' '.join(missing)}\n"
        "  Then pick the .venv interpreter in VS Code (kernel picker, top-right) and Run All.\n"
        "  Corporate proxy or PyPI blocked? See SETUP.md in the repo root.\n"
        f"  (pip said: {tail})\n"
    )

_ensure_packages([("anthropic", "anthropic")])
print("✓ Dependencies ready")
import os
# Make sure we are in the exercise folder (handles a kernel that starts at the repo root)
if not os.path.exists("Diagnose_Fix_Brief.py") and os.path.isdir("day1/04_diagnosing-ai-problems"):
    os.chdir("day1/04_diagnosing-ai-problems")

def _status(ok, msg):
    """Green/red banner in notebooks; plain text when run as a script."""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("not in a notebook kernel - use the plain-text banner")
        from IPython.display import display, HTML
        color = "#1a7f37" if ok else "#b42318"
        bg = "#e6f4ea" if ok else "#fdecea"
        icon = "✓" if ok else "✗"
        display(HTML(
            f'<div style="padding:12px 16px;border-radius:8px;background:{bg};'
            f'border:1.5px solid {color};color:{color};font-weight:600;'
            f'font-size:15px;font-family:sans-serif;">{icon} {msg}</div>'
        ))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

import os, pathlib

# ── API key via .env (gitignored — never committed) ──
# Your key lives in a .env file. We look for one next to this notebook or in any parent
# folder, so a single .env at the repo root can serve every exercise (paste once). If none
# exists yet, we create one from this template — fill it in, save, and re-run the cell.
_ENV_TEMPLATE = (
    "# Paste your Anthropic API key after the = (no quotes, no spaces), then save\n"
    "# and re-run the setup cell. Get a key at https://console.anthropic.com/\n"
    "# This file is gitignored — your key is never committed.\n"
    "ANTHROPIC_API_KEY=paste-your-key-here\n"
)

def _resolve_env_file():
    """Nearest existing .env walking up from the working dir (so one root .env serves every
    exercise); if none exists yet, point at the repo root — or this folder if the notebook
    was opened on its own."""
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents]
                 if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"

_env_file = _resolve_env_file()
if not _env_file.exists():
    _env_file.write_text(_ENV_TEMPLATE)
    print(f"Created {_env_file.name} in {_env_file.parent} — open it, paste your key after "
          "ANTHROPIC_API_KEY=, save, then re-run this cell.")

# Tiny .env parser (no python-dotenv dependency). Re-read on every run, so pasting your
# key and re-running picks it up. A real key in the environment (shell / Claude Code / CI)
# wins; the placeholder never sticks.
_file = {}
for _line in (_env_file.read_text().splitlines() if _env_file.exists() else []):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")
for _k, _v in _file.items():
    if _k != "ANTHROPIC_API_KEY":
        os.environ.setdefault(_k, _v)
_shell = os.environ.get("ANTHROPIC_API_KEY", "").strip()
api_key = _shell if _shell.startswith("sk-ant-") else _file.get("ANTHROPIC_API_KEY", "").strip()
if not api_key.startswith("sk-ant-"):
    print(
        "\n📋 Add your key to continue:\n"
        f"   1. Open this file:  {_env_file}\n"
        "   2. Replace  paste-your-key-here  with your key (it starts with sk-ant-)\n"
        "   3. Save the file, then click ▶ on this cell again.\n"
    )
else:
    import anthropic
    client = anthropic.Anthropic(api_key=api_key, timeout=30.0, max_retries=1)
    try:
        client.messages.create(model="claude-haiku-4-5", max_tokens=1,
                               messages=[{"role": "user", "content": "ping"}])
    except anthropic.AuthenticationError:
        _status(False, "That key was rejected. Run this cell again and paste the whole key (it starts with sk-ant-).")
        raise SystemExit("API key not accepted - re-run this cell and try again.")
    except Exception as exc:
        _status(False, "Could not reach the Claude API (" + type(exc).__name__ + "). Check your connection, then run this cell again.")
        raise
    else:
        os.environ["ANTHROPIC_API_KEY"] = api_key  # later cells (and any !python commands) pick it up from here
        _status(True, "API key verified - you're connected to Claude.")

## 1. The situation
Read Priya's email below, like it just hit your inbox (run the next cell to open it right here). She's losing faith in a support agent her team shipped three weeks ago, and the question she keeps getting from above is whether they bet on the wrong model. Give her a straight answer: what's actually wrong, whether it's fixable, and proof she can take upstairs.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("Priya_Email.md").read().split("---", 1)[-1]))

## 2. Get your baseline
Run the next cell. You'll get a RESOLVED rate and a cost for ticket T-4471. That rate is the pilot Priya's worried about, and it's your starting line. Write it down. Every scoreboard run is also recorded to `runs.jsonl` and replayed as a RUN HISTORY block under the scoreboard, so your baseline stays on screen for the rest of the exercise. (Give it a minute or two — it runs the ticket five times, and you'll see each trial land as it finishes.)

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5

## 3. Is it the model?
Priya's theory is that the model isn't good enough. Yours might be too. So test it head-on: the next cell runs the same eval on a bigger, pricier model. You'll get a new rate and a new cost. Then sit with two questions. Did the bigger model actually resolve the ticket? And what did it cost you to find out? Whatever you take away from that, you earned it. Nobody told you.

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5 --model claude-opus-4-8

## 4. Watch it work a ticket
The rate tells you it's failing. It doesn't tell you how. So watch one run start to finish. You'll see every move the coordinator made: what it looked up, which specialist it handed the ticket to, and the reply it sent the customer. Read that against the email. The customer raised more than one thing. How many did the agent take care of, and where did it stop?

Not sure what you're looking at? Drop the transcript into Claude and think out loud with it: *"Here's a trace from a support agent that closed a ticket the customer says isn't fixed. Walk me through what it did and where it might have gone wrong."* Treat what comes back as a lead to chase, not an answer to trust.

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --ticket T-4471

## 5. Go into the agents
Now you've got a theory about what went wrong. Time to look at the system that produced it. The "agents" aren't code. They're plain files sitting in this folder, and you open them right here in the editor.

Start with the coordinator, `system-prompt-coordinator.txt`. It's the one deciding who handles each ticket, so read how it's told to sort tickets and hand them off. After that, its tool list (`coordinator-tools.json`) and the specialists' own files are fair game.

What you're hunting for is an instruction, or a setup, that would make the agent behave exactly the way you just watched it behave. Something you could rewrite. If you're circling, ask Claude: *"Here are a coordinator's instructions. A ticket with two separate problems only got one of them handled. What in here could cause that?"*

## 6. Change one thing, measure again
Found something? Change it, save the file, and run the next cell. Did the rate move? The RUN HISTORY block under the scoreboard shows every run you've made — baseline included — so the before/after comparison is right there. If it didn't budge, your theory was off or only half-right, so go back to the trace. If it climbed, you found a real lever. Keep pulling until the agent resolves the ticket cleanly, run after run. Same model the whole way.

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5

## 7. Make sure it holds
It's easy to fix one ticket by accident. The next cell runs held-out tickets: different customers, different problems. If your fix resolves those too, it's real. If it doesn't, you tuned it to T-4471 and you've got more to do.

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --holdout --trials 3

## 8. Write Priya back
Now you can answer her. Fill in `client-brief-template.md`: what was actually breaking it in plain language, what you changed, the proof (the rate before and after, plus the cost), and the answer to the question she'll absolutely ask, *"why not just pay for a better model?"* You ran that test in step 3. You already know.